# Aggregation in Pandas
**Gary Mitchell**

**4/1/2021**

&nbsp;

Pandas makes *grouping* and *summarizing* data easy. However, the best way to get started
grouping data in Pandas is to see it in action.

We know that the baseball dataset is quite large, containing over 20,000 rows and 22 columns:

In [1]:
import pandas as pd

baseball = pd.read_csv("baseball.csv")
print(baseball.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21699 entries, 0 to 21698
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  21699 non-null  int64  
 1   id          21699 non-null  object 
 2   year        21699 non-null  int64  
 3   stint       21699 non-null  int64  
 4   team        21699 non-null  object 
 5   lg          21634 non-null  object 
 6   g           21699 non-null  int64  
 7   ab          21699 non-null  int64  
 8   r           21699 non-null  int64  
 9   h           21699 non-null  int64  
 10  X2b         21699 non-null  int64  
 11  X3b         21699 non-null  int64  
 12  hr          21699 non-null  int64  
 13  rbi         21687 non-null  float64
 14  sb          21449 non-null  float64
 15  cs          17174 non-null  float64
 16  bb          21699 non-null  int64  
 17  so          20394 non-null  float64
 18  ibb         14171 non-null  float64
 19  hbp         21322 non-nul

## Creating a subset of a dataframe

Now, to make things easy, let's create a small subset of the complete dataframe. We'll
extract the data for Babe Ruth and Willy Mays (player ids ruthba01 and mayswi01). We'll
use a logical index to extract only the rows we want.

In the following example,

    baseball['id'].isin(players)

will be a boolean series with a value for each row in the
dataframe. The value will be True if the player id is either 'ruthba01' or 'mayswi01' and
False otherwise. We then use the boolean series as the index in:

    baseball.loc[...]

The result is that we quickly extracted the 45 rows for Babe Ruth and Willy Mays.

In [2]:
players = ['ruthba01', 'mayswi01']
maysruthall = baseball.loc[baseball['id'].isin(players)]

print(maysruthall)


       Unnamed: 0        id  year  stint team  lg    g   ab    r    h  ...  \
2541        14646  ruthba01  1914      1  BOS  AL    5   10    1    2  ...   
2655        15457  ruthba01  1915      1  BOS  AL   42   92   16   29  ...   
2789        16238  ruthba01  1916      1  BOS  AL   67  136   18   37  ...   
2911        16776  ruthba01  1917      1  BOS  AL   52  123   14   40  ...   
3019        17286  ruthba01  1918      1  BOS  AL   95  317   50   95  ...   
3132        17790  ruthba01  1919      1  BOS  AL  130  432  103  139  ...   
3256        18329  ruthba01  1920      1  NYA  AL  142  457  158  172  ...   
3373        18834  ruthba01  1921      1  NYA  AL  152  540  177  204  ...   
3491        19363  ruthba01  1922      1  NYA  AL  110  406   94  128  ...   
3610        19883  ruthba01  1923      1  NYA  AL  152  522  151  205  ...   
3740        20420  ruthba01  1924      1  NYA  AL  153  529  143  200  ...   
3879        20967  ruthba01  1925      1  NYA  AL   98  359   61

## Dropping unneeded columns

Looking at the subset, we have some columns that aren't going to do us any good. The "Unnamed: 0"
column is simply a sequential id that is a carryover from the original R dataset used to
create baseball.csv. However, we also have some columns with missing data. Looking at the
data for the subset, we see that Babe Ruth's records are all missing the following metrics:

- cs
- ibb
- sf
- gidp

We can infer that the keeping of statistics on these metrics did not begin until after Babe Ruth
retired from baseball. So, we'll drop these columns from our subset to simplify subsequent
processing:

In [3]:
maysruth = maysruthall.drop(columns=['Unnamed: 0','cs', 'ibb', 'sf', 'gidp'])
print(maysruth)

             id  year  stint team  lg    g   ab    r    h  X2b  X3b  hr  \
2541   ruthba01  1914      1  BOS  AL    5   10    1    2    1    0   0   
2655   ruthba01  1915      1  BOS  AL   42   92   16   29   10    1   4   
2789   ruthba01  1916      1  BOS  AL   67  136   18   37    5    3   3   
2911   ruthba01  1917      1  BOS  AL   52  123   14   40    6    3   2   
3019   ruthba01  1918      1  BOS  AL   95  317   50   95   26   11  11   
3132   ruthba01  1919      1  BOS  AL  130  432  103  139   34   12  29   
3256   ruthba01  1920      1  NYA  AL  142  457  158  172   36    9  54   
3373   ruthba01  1921      1  NYA  AL  152  540  177  204   44   16  59   
3491   ruthba01  1922      1  NYA  AL  110  406   94  128   24    8  35   
3610   ruthba01  1923      1  NYA  AL  152  522  151  205   45   13  41   
3740   ruthba01  1924      1  NYA  AL  153  529  143  200   39    7  46   
3879   ruthba01  1925      1  NYA  AL   98  359   61  104   12    2  25   
4015   ruthba01  1926    

## Aggregation the hard way
That looks better! Now we have cleaner data to work with. Now, let's see some examples of
using Pandas aggregation. Let's say that we want to determine the last year in which
each of these players played.

We *could* do the following:

In [4]:
years = {p: maysruth.loc[maysruth['id'] == p]['year'].max() for p in players}
print(years)

{'ruthba01': 1935, 'mayswi01': 1973}


## Aggregation using groupby
The problem is that this is kind of ugly/messy and won't be very efficient. Enter Pandas'
groupby method. Here we are asking pandas to group the data by player.

In [5]:
byplayer = maysruth.groupby('id')
for p in byplayer:
    print(p)

('mayswi01',              id  year  stint team  lg    g   ab    r    h  X2b  X3b  hr  \
7131   mayswi01  1951      1  NY1  NL  121  464   59  127   22    5  20   
7274   mayswi01  1952      1  NY1  NL   34  127   17   30    2    4   4   
7555   mayswi01  1954      1  NY1  NL  151  565  119  195   33   13  41   
7719   mayswi01  1955      1  NY1  NL  152  580  123  185   18   13  51   
7887   mayswi01  1956      1  NY1  NL  152  578  101  171   27    8  36   
8057   mayswi01  1957      1  NY1  NL  152  585  112  195   26   20  35   
8229   mayswi01  1958      1  SFN  NL  152  600  121  208   33   11  29   
8408   mayswi01  1959      1  SFN  NL  151  575  125  180   43    5  34   
8605   mayswi01  1960      1  SFN  NL  153  595  107  190   29   12  29   
8811   mayswi01  1961      1  SFN  NL  154  572  129  176   32    3  40   
8994   mayswi01  1962      1  SFN  NL  162  621  130  189   36    5  49   
9194   mayswi01  1963      1  SFN  NL  157  596  115  187   32    7  38   
9407   maysw

Hmmm. That's interesting, the result looks like we created *two* subsets of maysruth:

- One with Babe Ruth's data
- One with Willy Mays' data.

Now using these *by groups* we can easily find the last year for each player:

In [6]:
byplayer['year'].max()

id
mayswi01    1973
ruthba01    1935
Name: year, dtype: int64

Wow! It doesn't get much easier than that!! What if we want to calculate the total runs
scored for each player?

In [7]:
byplayer['r'].sum()

id
mayswi01    2062
ruthba01    2174
Name: r, dtype: int64

How about total homeruns (hr) and stolen bases (sb)?

In [8]:
byplayer[['r','sb']].sum()

,r,sb
id,,
mayswi01,2062,338.0
ruthba01,2174,123.0


123 stolen bases for Babe Ruth? Who'd have thought?

## Aggregating multiple columns in a dataframe

Now what if we want to calculate career totals on ALL of the metrics by player? This is
a little harder, but not much...

maysruth contains some columns that don't make sense to sum, either because they're non-numeric
(e.g. team) or because summing them simply doesn't make sense (e.g. year, stint). So, in
order to obtain a sensible aggregation, we should drop those columns (at least temporarily),
then we can sum over all of the remaining columns:

In [9]:
byplayer = maysruth.drop(columns=['year','stint','team','lg']).groupby('id')
byplayer.sum()

,g,ab,r,h,X2b,X3b,hr,rbi,sb,bb,so,hbp,sh
id,,,,,,,,,,,,,
mayswi01,2992,10881,2062,3283,523,140,660,1903.0,338.0,1464,1526.0,44.0,13.0
ruthba01,2503,8398,2174,2873,506,136,714,2217.0,123.0,2062,1330.0,43.0,113.0


If we save the result and look at the type, we'll see that we created a new dataframe:

In [10]:
career = byplayer.sum()
print(career.info())

<class 'pandas.core.frame.DataFrame'>
Index: 2 entries, mayswi01 to ruthba01
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   g       2 non-null      int64  
 1   ab      2 non-null      int64  
 2   r       2 non-null      int64  
 3   h       2 non-null      int64  
 4   X2b     2 non-null      int64  
 5   X3b     2 non-null      int64  
 6   hr      2 non-null      int64  
 7   rbi     2 non-null      float64
 8   sb      2 non-null      float64
 9   bb      2 non-null      int64  
 10  so      2 non-null      float64
 11  hbp     2 non-null      float64
 12  sh      2 non-null      float64
dtypes: float64(5), int64(8)
memory usage: 224.0+ bytes
None


We can see columns for all of the stats, but where did the 'id' column go? We saw it
in the output, but it doesn't
appear to be defined as a column in the dataframe. That's because Pandas set the id (our
grouping variable) as the dataframe index:

In [11]:
career.index

Index(['mayswi01', 'ruthba01'], dtype='object', name='id')

## Summarizing by multiple group by variables

In looking at the data for Willy Mays, it appears that he played for two different teams in 1972.
If we wanted to *accurately* calculate single season statistics for the players, we'd have to
summarize by player and year. Pandas makes it easy to do this:

In [12]:
season = maysruth.drop(columns=['stint','team','lg']).groupby(['id','year']).sum()
season

g   ab    r    h  X2b  X3b  hr    rbi    sb   bb     so  hbp  \
id       year                                                                   
mayswi01 1951  121  464   59  127   22    5  20   68.0   7.0   57   60.0  2.0   
         1952   34  127   17   30    2    4   4   23.0   4.0   16   17.0  1.0   
         1954  151  565  119  195   33   13  41  110.0   8.0   66   57.0  2.0   
         1955  152  580  123  185   18   13  51  127.0  24.0   79   60.0  4.0   
         1956  152  578  101  171   27    8  36   84.0  40.0   68   65.0  1.0   
         1957  152  585  112  195   26   20  35   97.0  38.0   76   62.0  1.0   
         1958  152  600  121  208   33   11  29   96.0  31.0   78   56.0  1.0   
         1959  151  575  125  180   43    5  34  104.0  27.0   65   58.0  2.0   
         1960  153  595  107  190   29   12  29  103.0  25.0   61   70.0  4.0   
         1961  154  572  129  176   32    3  40  123.0  18.0   81   77.0  2.0   
         1962  162  621  130  189   36    5  49  141.0  18.0   78   85.0  4.0   
         1963  157  596  115  187   32    7  38  103.0   8.0   66   83.0  2.0   
         1964  157  578  121  171   21    9  47  111.0  19.0   82   72.0  1.0   
         1965  157  558  118  177   21    3  52  112.0   9.0   76   71.0  0.0   
         1966  152  552   99  159   29    4  37  103.0   5.0   70   81.0  2.0   
         1967  141  486   83  128   22    2  22   70.0   6.0   51   92.0  2.0   
         1968  148  498   84  144   20    5  23   79.0  12.0   67   81.0  2.0   
         1969  117  403   64  114   17    3  13   58.0   6.0   49   71.0  3.0   
         1970  139  478   94  139   15    2  28   83.0   5.0   79   90.0  3.0   
         1971  136  417   82  113   24    5  18   61.0  23.0  112  123.0  3.0   
         1972   88  244   35   61   11    1   8   22.0   4.0   60   48.0  1.0   
         1973   66  209   24   44   10    0   6   25.0   1.0   27   47.0  1.0   
ruthba01 1914    5   10    1    2    1    0   0    2.0   0.0    0    4.0  0.0   
         1915   42   92   16   29   10    1   4   21.0   0.0    9   23.0  0.0   
         1916   67  136   18   37    5    3   3   15.0   0.0   10   23.0  0.0   
         1917   52  123   14   40    6    3   2   12.0   0.0   12   18.0  0.0   
         1918   95  317   50   95   26   11  11   66.0   6.0   58   58.0  2.0   
         1919  130  432  103  139   34   12  29  114.0   7.0  101   58.0  6.0   
         1920  142  457  158  172   36    9  54  137.0  14.0  150   80.0  3.0   
         1921  152  540  177  204   44   16  59  171.0  17.0  145   81.0  4.0   
         1922  110  406   94  128   24    8  35   99.0   2.0   84   80.0  1.0   
         1923  152  522  151  205   45   13  41  131.0  17.0  170   93.0  4.0   
         1924  153  529  143  200   39    7  46  121.0   9.0  142   81.0  4.0   
         1925   98  359   61  104   12    2  25   66.0   2.0   59   68.0  2.0   
         1926  152  495  139  184   30    5  47  150.0  11.0  144   76.0  3.0   
         1927  151  540  158  192   29    8  60  164.0   7.0  137   89.0  0.0   
         1928  154  536  163  173   29    8  54  142.0   4.0  137   87.0  3.0   
         1929  135  499  121  172   26    6  46  154.0   5.0   72   60.0  3.0   
         1930  145  518  150  186   28    9  49  153.0  10.0  136   61.0  1.0   
         1931  145  534  149  199   31    3  46  163.0   5.0  128   51.0  1.0   
         1932  133  457  120  156   13    5  41  137.0   2.0  130   62.0  2.0   
         1933  137  459   97  138   21    3  34  103.0   4.0  114   90.0  2.0   
         1934  125  365   78  105   17    4  22   84.0   1.0  104   63.0  2.0   
         1935   28   72   13   13    0    0   6   12.0   0.0   20   24.0  0.0   

                 sh  
id       year        
mayswi01 1951   1.0  
         1952   0.0  
         1954   0.0  
         1955   0.0  
         1956   0.0  
         1957   0.0  
         1958   0.0  
         1959   0.0  
         1960   0.0  
         1961   0.0  
         1962   0.0  
    

**Note: for the Data Manipulation project, you do not have to summarize by year. Per the
assignment instructions, you can assume that each record for a player represents a
separate year. It's not accurate, but it simplifies the assignment for you.**

In this case, Pandas created a *multiindex* (i.e. a hierarchical index). When we look in the output we
see all of the rows for Willy Mays grouped together, by year, with only one row per year. Because we aggregated
using sum(), the values in a row (i.e. for a year) represent the sum of all records for that player for that year!

There are some nifty things that we can do with multiindex index operations too. However,
The break in years between Willy's and Babe's records prevents a demonstration of these
kinds of operations on this particular data subset.

However, we *can* access all of the rows for a player

In [13]:
season.loc['mayswi01']

,g,ab,r,h,X2b,X3b,hr,rbi,sb,bb,so,hbp,sh
year,,,,,,,,,,,,,
1951,121,464,59,127,22,5,20,68.0,7.0,57,60.0,2.0,1.0
1952,34,127,17,30,2,4,4,23.0,4.0,16,17.0,1.0,0.0
1954,151,565,119,195,33,13,41,110.0,8.0,66,57.0,2.0,0.0
1955,152,580,123,185,18,13,51,127.0,24.0,79,60.0,4.0,0.0
1956,152,578,101,171,27,8,36,84.0,40.0,68,65.0,1.0,0.0
1957,152,585,112,195,26,20,35,97.0,38.0,76,62.0,1.0,0.0
1958,152,600,121,208,33,11,29,96.0,31.0,78,56.0,1.0,0.0
1959,151,575,125,180,43,5,34,104.0,27.0,65,58.0,2.0,0.0
1960,153,595,107,190,29,12,29,103.0,25.0,61,70.0,4.0,0.0


or even a single year:

In [14]:
season.loc['mayswi01', 1965]

g      157.0
ab     558.0
r      118.0
h      177.0
X2b     21.0
X3b      3.0
hr      52.0
rbi    112.0
sb       9.0
bb      76.0
so      71.0
hbp      0.0
sh       2.0
Name: (mayswi01, 1965), dtype: float64

We can also access a one or more columns. Watch what happens:

In [15]:
season[['g','hr']]

g  hr
id       year         
mayswi01 1951  121  20
         1952   34   4
         1954  151  41
         1955  152  51
         1956  152  36
         1957  152  35
         1958  152  29
         1959  151  34
         1960  153  29
         1961  154  40
         1962  162  49
         1963  157  38
         1964  157  47
         1965  157  52
         1966  152  37
         1967  141  22
         1968  148  23
         1969  117  13
         1970  139  28
         1971  136  18
         1972   88   8
         1973   66   6
ruthba01 1914    5   0
         1915   42   4
         1916   67   3
         1917   52   2
         1918   95  11
         1919  130  29
         1920  142  54
         1921  152  59
         1922  110  35
         1923  152  41
         1924  153  46
         1925   98  25
         1926  152  47
         1927  151  60
         1928  154  54
         1929  135  46
         1930  145  49
         1931  145  46
         1932  133  41
         1933  137  34
         1934  125  22
         1935   28   6

Really cool, right? Our multiindex was preserved.

Let's determine the number of years played by each player. The counts will be the same
on all columns, so we just *pick* one. We specify level='id' to indicate that we want to
count all of the rows for an 'id' value.

In [16]:
season['g'].count(level='id')

id
mayswi01    22
ruthba01    22
Name: g, dtype: int64

What a coincidence! It turns out that Babe Ruth and Willy Mays both played for 22 seasons!!

## Flattening a dataframe

Now, what if we want to get rid of the multiindex and just have
a normal dataframe? We can do that too!

In [17]:
season.reset_index()

,id,year,g,ab,r,h,X2b,X3b,hr,rbi,sb,bb,so,hbp,sh
0,mayswi01,1951,121,464,59,127,22,5,20,68.0,7.0,57,60.0,2.0,1.0
1,mayswi01,1952,34,127,17,30,2,4,4,23.0,4.0,16,17.0,1.0,0.0
2,mayswi01,1954,151,565,119,195,33,13,41,110.0,8.0,66,57.0,2.0,0.0
3,mayswi01,1955,152,580,123,185,18,13,51,127.0,24.0,79,60.0,4.0,0.0
4,mayswi01,1956,152,578,101,171,27,8,36,84.0,40.0,68,65.0,1.0,0.0
5,mayswi01,1957,152,585,112,195,26,20,35,97.0,38.0,76,62.0,1.0,0.0
6,mayswi01,1958,152,600,121,208,33,11,29,96.0,31.0,78,56.0,1.0,0.0
7,mayswi01,1959,151,575,125,180,43,5,34,104.0,27.0,65,58.0,2.0,0.0
8,mayswi01,1960,153,595,107,190,29,12,29,103.0,25.0,61,70.0,4.0,0.0
9,mayswi01,1961,154,572,129,176,32,3,40,123.0,18.0,81,77.0,2.0,0.0


Now we have a standard dataframe with a single index, not related to data. Of course,
we see the 'id' repeated multiple times for each player. But now we can select on
the values of the columns.
